# 03: ridge regression

**The algorithm.** Linear regression with a penalty on large weights. It has an exact answer: solve `(XᵀX + αI) w = Xᵀy`.

**Where the time goes.** Almost all of it is building `XᵀX`: one multiply of the n × d data with itself. The solve afterwards is a d × d problem, which is tiny. This is the *most matrix-math-bound* algorithm in the set, so the question is simply: who drives the multiply hardware best?

**Why there is no Rust engine.** Rust would hand the multiply to the same BLAS routine NumPy uses. There's nothing for compiled code to fuse or speed up. That is itself a finding: not every algorithm has headroom.

**Our trick, and the accuracy trap it nearly fell into.** scikit-learn centers a full copy of X before multiplying. Our engines never copy X: it centers one cache-sized chunk at a time and adds each chunk's result into a 64-bit running total.

A first version took a bigger shortcut, computing `XᵀX` on the raw data and subtracting the mean afterwards. It was fast but lost precision in 32-bit floats: its coefficients were up to 7× further from the exact answer than scikit-learn's. The chunked version fixed that. So this notebook checks **accuracy as well as speed**: every engine, scikit-learn included, is compared against an exact answer computed in 64-bit floats.

In [ ]:
import sys
sys.path.insert(0, ".")  # so bench_utils (next to this notebook) imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_in_rust as mr
from bench_utils import (machine_info, timeit, load_covtype, add_speedup,
                         save_results, line_chart, fmt_ms, only_available)

pd.set_option("display.float_format", "{:,.4f}".format)
%matplotlib inline
pd.Series(machine_info())

In [ ]:
import os
QUICK = os.environ.get("QUICK") == "1"  # QUICK=1 ./run_all.sh for a fast smoke run
from sklearn.linear_model import Ridge as SkRidge

def make_data(n, d, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d), dtype=np.float32) * np.linspace(0.5, 3, d, dtype=np.float32) + 2
    w = rng.standard_normal(d).astype(np.float32)
    y = X @ w + 1.5 + rng.standard_normal(n, dtype=np.float32)
    return X, y

def exact_ridge(X, y, alpha=1.0, chunk=200_000):
    # Reference answer in float64, chunked so it never needs a full float64 copy.
    n, d = X.shape
    mean = X.sum(0, dtype=np.float64) / n
    y_mean = y.sum(dtype=np.float64) / n
    G, b = np.zeros((d, d)), np.zeros(d)
    for s in range(0, n, chunk):
        c = X[s:s + chunk].astype(np.float64) - mean
        G += c.T @ c
        b += c.T @ (y[s:s + chunk].astype(np.float64) - y_mean)
    return np.linalg.solve(G + alpha * np.eye(d), b)

def race(n, d):
    X, y = make_data(n, d)
    exact = exact_ridge(X, y)
    models = only_available({
        "sklearn": lambda: SkRidge(alpha=1.0),
        "numpy": lambda: mr.Ridge(alpha=1.0, backend="numpy"),
        "mlx": lambda: mr.Ridge(alpha=1.0, backend="mlx"),
    })
    out, ref = [], None
    for name, make in models.items():
        fitted = {}
        def run():
            fitted["m"] = make().fit(X, y)
        t = timeit(run, repeats=3)
        m = fitted["m"]
        if name == "sklearn":
            ref = m
        out.append({"rows": n, "features": d, "engine": name, "fit s": t,
                    "max error vs exact": float(np.abs(m.coef_ - exact).max())})
    return out

## Race 1: more rows (100 features)

In [ ]:
SIZES = [100_000, 500_000] if QUICK else [100_000, 1_000_000, 4_000_000]
by_rows = add_speedup(pd.DataFrame([r for n in SIZES for r in race(n, 100)]), "fit s", ["rows"])
line_chart(by_rows, "rows", "fit s", "Ridge: fit time as rows grow (100 features)", "rows", "seconds (log scale)")
plt.show()
by_rows

## Race 2: more features (1M rows)

The multiply grows with d², so wider data shifts more of the work onto the matrix hardware.

In [ ]:
n = 200_000 if QUICK else 1_000_000
by_d = add_speedup(pd.DataFrame([r for d in [20, 100, 500] for r in race(n, d)]), "fit s", ["features"])
line_chart(by_d, "features", "fit s", f"Ridge: fit time as features grow ({n:,} rows)", "features", "seconds (log scale)")
plt.show()
by_d

## Accuracy: how far is each engine from the exact answer?

Largest error in any coefficient, against a 64-bit reference. Smaller is better. All three engines work in 32-bit floats, so none is exact; what matters is how errors grow with more rows.

In [ ]:
both = pd.concat([by_rows, by_d]).drop_duplicates(["rows", "features", "engine"])
both.pivot_table(index=["rows", "features"], columns="engine", values="max error vs exact").map(lambda v: f"{v:.1e}")

In [ ]:
save_results("ridge", by_rows, notes={"by_features": by_d.to_dict(orient="records")})
both.pivot_table(index=["rows", "features"], columns="engine", values="vs sklearn").round(2)